### prompt chaining

In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [2]:
# create a state graph for prompt chaining
class LLMState(TypedDict):
    input_text: str
    outlier_text: str
    output_text: str

In [4]:
load_dotenv()  # Load environment variables from .env file
LLM = ChatGoogleGenerativeAI(model="gemini-3.8-flash", api_key=os.getenv("GOOGLE_API_KEY"))

In [5]:
# create a node for generating an outlier based on the input text
def generate_outlier(state: LLMState) -> LLMState:
    outlier_prompt = f"Generate an outlier for the following input: {state['input_text']}"
    outlier_response = LLM.invoke(outlier_prompt).content
    state['outlier_text'] = outlier_response
    return state

In [6]:
# create a node for generating a blog using input text and outlier
def generate_blog(state: LLMState) -> LLMState:
    blog_prompt = f"Write a blog using the input: {state['input_text']} and the outlier: {state['outlier_text']}"
    blog_response = LLM.invoke(blog_prompt).content
    state['output_text'] = blog_response
    return state

In [12]:
# create graph
graph = StateGraph(LLMState)

# add nodes to the graph
graph.add_node('generate_outlier', generate_outlier)
graph.add_node('generate_blog', generate_blog)

# add edges to the graph
graph.add_edge(START, 'generate_outlier')
graph.add_edge('generate_outlier', 'generate_blog')
graph.add_edge('generate_blog', END)

# compile the graph into a workflow
workflow = graph.compile()  

In [13]:
# execute the state graph
initial_state = {'input_text': "Virat Kohli"}
final_state = workflow.invoke(initial_state)
print(final_state['output_text'])  # Output the final blog generated by the LLM

[{'type': 'text', 'text': '# The King and the Anomaly: Virat Kohli and the Anatomy of an Outlier\n\nWhen you hear the name **Virat Kohli**, your mind instinctively lands on numbers that defy the law of averages. Over 26,000 international runs, 80 centuries, and an insatiable appetite for the chase. In the language of mathematics, Kohli is practically the definition of a positive statistical anomaly—a man living permanently at the extreme end of the bell curve.\n\nYet, when you view Kohli through the lens of an **"outlier"**, the conversation takes unexpected, fascinating turns. From bizarre quirks of cricket trivia to deep philosophical contrasts in world sport, exploring Kohli as—and alongside—an outlier tells us a lot about the peculiar beauty of athletic greatness.\n\nHere are two distinct ways to explore Virat Kohli and the outlier.\n\n---\n\n## 1. The Statistical Glitch: The Man Who Took a Wicket on Ball "Zero"\n\nCricket is a sport governed by pedantic scorecards, but on August 3